## Q1: Spark version

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")  # use all cores in the local machine
    .appName("homework")
    .getOrCreate()
)
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 00:35:47 WARN Utils: Your hostname, Pavel-Kalmykovs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface en0)
26/03/10 00:35:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 00:35:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.1.1'

## Q2: Average Parquet file size

In [2]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.repartition(4).write.mode("overwrite").parquet("output/")

In [3]:
from pathlib import Path

files = list(Path("output").glob("*.parquet"))
avg_mb = sum(f.stat().st_size for f in files) / len(files) / 1024 / 1024
print(f"{avg_mb:.1f} MB")

24.4 MB


## Q3: Trips on November 15

In [4]:
df.createOrReplaceTempView("trips")

spark.sql("""
  SELECT COUNT(*)
  FROM trips
  -- == "2025-11-15" would only pick up 2025-11-15 00:00:00
  WHERE tpep_pickup_datetime >= '2025-11-15'
    AND tpep_pickup_datetime < '2025-11-16'
""").show(5, truncate=False)

+--------+
|count(1)|
+--------+
|162604  |
+--------+



## Q4: Longest trip in hours

In [5]:
spark.sql("""
  SELECT MAX(
      ROUND(
          (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600,
          2
      )
  ) AS max_hours
  FROM trips
""").show(5, truncate=False)

+---------+
|max_hours|
+---------+
|90.65    |
+---------+



## Q5: Spark UI port

In [6]:
# The Spark UI runs on port 4040
# Visit http://localhost:4040 while a SparkSession is active

## Q6: Least frequent pickup zone

In [7]:
zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")
zones.createOrReplaceTempView("zones")

spark.sql("""
  SELECT z.Zone, COUNT(*) AS cnt
  FROM trips t
  JOIN zones z ON t.PULocationID = z.LocationID
  GROUP BY z.Zone
  ORDER BY cnt ASC
""").show(5, truncate=False)

+---------------------------------------------+---+
|Zone                                         |cnt|
+---------------------------------------------+---+
|Governor's Island/Ellis Island/Liberty Island|1  |
|Eltingville/Annadale/Prince's Bay            |1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
+---------------------------------------------+---+
only showing top 5 rows
